In [1]:
#Gold Layer Transformation

#loading the fact table from silver table 

df_fact = spark.table("silver_f_sls_t")

df_fact.printSchema()
df_fact.show(5)

StatementMeta(, 2ec5e924-7aa4-4595-b3ed-29dece6154fd, 3, Finished, Available, Finished, False)

root
 |-- trxn_id: integer (nullable = true)
 |-- prod_num: string (nullable = true)
 |-- store_num: string (nullable = true)
 |-- cust_num: string (nullable = true)
 |-- trxn_dt: timestamp (nullable = true)
 |-- qty: integer (nullable = true)
 |-- amount: decimal(11,2) (nullable = true)
 |-- day: integer (nullable = true)
 |-- silver_loaded_at: timestamp (nullable = true)

+-------+--------+---------+--------+-------------------+---+------+--------+--------------------+
|trxn_id|prod_num|store_num|cust_num|            trxn_dt|qty|amount|     day|    silver_loaded_at|
+-------+--------+---------+--------+-------------------+---+------+--------+--------------------+
|    415|    P003|     S001|    C045|2026-07-28 22:17:55|  2| 87.90|20260731|2026-08-12 09:58:...|
|    416|    P006|     S001|    C064|2026-07-28 18:35:58|  4|146.12|20260731|2026-08-12 09:58:...|
|    417|    P001|     S001|    C065|2026-07-28 17:38:08|  1| 62.49|20260731|2026-08-12 09:58:...|
|    418|    P033|     S001| 

In [6]:
from pyspark.sql import functions as F

df_fact = spark.table("silver_f_sls_t")


df_gold_sales_by_product = (
    df_fact
    .withColumn("summary_date", F.to_date(F.col("trxn_dt")))
    .groupBy("prod_num", "summary_date")
    .agg(
        F.sum("qty").alias("total_qty"),
        F.sum("amount").alias("total_amount"),
        F.count("trxn_id").alias("txn_count")
    )
    .orderBy("summary_date", "prod_num")
)

df_gold_sales_by_product = df_gold_sales_by_product.withColumn("gold_loaded_at", F.current_timestamp())

df_gold_sales_by_product.show(10)

StatementMeta(, 2ec5e924-7aa4-4595-b3ed-29dece6154fd, 17, Finished, Available, Finished, False)

+--------+------------+---------+------------+---------+--------------------+
|prod_num|summary_date|total_qty|total_amount|txn_count|      gold_loaded_at|
+--------+------------+---------+------------+---------+--------------------+
|    P001|  2026-07-28|       34|     1539.30|       12|2026-08-12 10:13:...|
|    P003|  2026-07-28|       22|     1334.76|       10|2026-08-12 10:13:...|
|    P004|  2026-07-28|       20|     1205.84|        6|2026-08-12 10:13:...|
|    P005|  2026-07-28|       14|      527.40|        8|2026-08-12 10:13:...|
|    P006|  2026-07-28|       20|      814.00|        6|2026-08-12 10:13:...|
|    P007|  2026-07-28|        8|       52.72|        2|2026-08-12 10:13:...|
|    P008|  2026-07-28|        6|      548.72|        4|2026-08-12 10:13:...|
|    P010|  2026-07-28|       22|      544.90|        6|2026-08-12 10:13:...|
|    P011|  2026-07-28|        8|      612.40|        2|2026-08-12 10:13:...|
|    P012|  2026-07-28|        8|      694.08|        2|2026-08-

In [7]:
#writing as delta table 
df_gold_sales_by_product.write.format("delta").mode("overwrite").saveAsTable("gold_sales_by_product")

StatementMeta(, 2ec5e924-7aa4-4595-b3ed-29dece6154fd, 19, Finished, Available, Finished, False)

In [8]:
#gold transformation for customer

from pyspark.sql import functions as F

df_gold_sales_by_customer = (
    df_fact
    .withColumn("summary_date", F.to_date(F.col("trxn_dt")))
    .groupBy("cust_num", "summary_date")
    .agg(
        F.sum("qty").alias("total_qty"),
        F.sum("amount").alias("total_amount"),
        F.count("trxn_id").alias("txn_count")
    )
    .orderBy("summary_date", "cust_num")
)

df_gold_sales_by_customer = df_gold_sales_by_customer.withColumn("gold_loaded_at", F.current_timestamp())

df_gold_sales_by_customer.show(10)

StatementMeta(, 2ec5e924-7aa4-4595-b3ed-29dece6154fd, 35, Finished, Available, Finished, False)

+--------+------------+---------+------------+---------+--------------------+
|cust_num|summary_date|total_qty|total_amount|txn_count|      gold_loaded_at|
+--------+------------+---------+------------+---------+--------------------+
|    C002|  2026-07-28|        8|      529.44|        2|2026-08-12 10:14:...|
|    C004|  2026-07-28|        6|      259.06|        4|2026-08-12 10:14:...|
|    C005|  2026-07-28|       14|      747.72|        4|2026-08-12 10:14:...|
|    C006|  2026-07-28|        6|      304.26|        2|2026-08-12 10:14:...|
|    C007|  2026-07-28|        2|      161.30|        2|2026-08-12 10:14:...|
|    C008|  2026-07-28|        2|      137.80|        2|2026-08-12 10:14:...|
|    C010|  2026-07-28|        6|      130.38|        2|2026-08-12 10:14:...|
|    C011|  2026-07-28|        4|      244.44|        2|2026-08-12 10:14:...|
|    C014|  2026-07-28|        6|      517.62|        2|2026-08-12 10:14:...|
|    C017|  2026-07-28|        8|      234.64|        4|2026-08-

In [9]:
df_gold_sales_by_customer.write.format("delta").mode("overwrite").saveAsTable("gold_sales_by_customer")

StatementMeta(, 2ec5e924-7aa4-4595-b3ed-29dece6154fd, 37, Finished, Available, Finished, False)

In [11]:
#Sanity check

df_check = spark.table("gold_sales_by_customer")
print("Row count:", df_check.count())
df_check.show(5)

spark.table("silver_f_sls_t").filter(
    "cust_num = 'C002' AND day = 20260731"
).agg(F.sum("amount").alias("check_total")).show()

StatementMeta(, 2ec5e924-7aa4-4595-b3ed-29dece6154fd, 52, Finished, Available, Finished, False)

Row count: 520
+--------+------------+---------+------------+---------+--------------------+
|cust_num|summary_date|total_qty|total_amount|txn_count|      gold_loaded_at|
+--------+------------+---------+------------+---------+--------------------+
|    C002|  2026-07-28|        8|      529.44|        2|2026-08-12 10:14:...|
|    C004|  2026-07-28|        6|      259.06|        4|2026-08-12 10:14:...|
|    C005|  2026-07-28|       14|      747.72|        4|2026-08-12 10:14:...|
|    C006|  2026-07-28|        6|      304.26|        2|2026-08-12 10:14:...|
|    C007|  2026-07-28|        2|      161.30|        2|2026-08-12 10:14:...|
+--------+------------+---------+------------+---------+--------------------+
only showing top 5 rows

+-----------+
|check_total|
+-----------+
|     529.44|
+-----------+



In [12]:
#gold transformation for store

from pyspark.sql import functions as F

df_gold_sales_by_store = (
    df_fact
    .withColumn("summary_date", F.to_date(F.col("trxn_dt")))
    .groupBy("store_num", "summary_date")
    .agg(
        F.sum("qty").alias("total_qty"),
        F.sum("amount").alias("total_amount"),
        F.count("trxn_id").alias("txn_count")
    )
    .orderBy("summary_date", "store_num")
)

df_gold_sales_by_store = df_gold_sales_by_store.withColumn("gold_loaded_at", F.current_timestamp())

df_gold_sales_by_store.show(10)

StatementMeta(, 2ec5e924-7aa4-4595-b3ed-29dece6154fd, 62, Finished, Available, Finished, False)

+---------+------------+---------+------------+---------+--------------------+
|store_num|summary_date|total_qty|total_amount|txn_count|      gold_loaded_at|
+---------+------------+---------+------------+---------+--------------------+
|     S001|  2026-07-28|       84|     4906.52|       40|2026-08-12 10:16:...|
|     S002|  2026-07-28|      168|     8485.40|       60|2026-08-12 10:16:...|
|     S003|  2026-07-28|      140|     7021.70|       54|2026-08-12 10:16:...|
|     S004|  2026-07-28|      116|     6767.90|       48|2026-08-12 10:16:...|
|     S005|  2026-07-28|      166|     9930.02|       64|2026-08-12 10:16:...|
|     S001|  2026-08-04|      145|     7682.00|       57|2026-08-12 10:16:...|
|     S002|  2026-08-04|      171|     8091.83|       69|2026-08-12 10:16:...|
|     S003|  2026-08-04|      166|     9362.19|       62|2026-08-12 10:16:...|
|     S004|  2026-08-04|      159|     7140.61|       66|2026-08-12 10:16:...|
|     S005|  2026-08-04|      152|     7784.81|     

In [13]:
df_gold_sales_by_store.write.format("delta").mode("overwrite").saveAsTable("gold_sales_by_store")

StatementMeta(, 2ec5e924-7aa4-4595-b3ed-29dece6154fd, 64, Finished, Available, Finished, False)

In [15]:
#Sanity check

df_check = spark.table("gold_sales_by_store")
print("Row count:", df_check.count())
df_check.show(5)

spark.table("silver_f_sls_t").filter(
    "store_num = 'S002' AND day = 20260731"
).agg(F.sum("amount").alias("check_total")).show()

StatementMeta(, 2ec5e924-7aa4-4595-b3ed-29dece6154fd, 78, Finished, Available, Finished, False)

Row count: 30
+---------+------------+---------+------------+---------+--------------------+
|store_num|summary_date|total_qty|total_amount|txn_count|      gold_loaded_at|
+---------+------------+---------+------------+---------+--------------------+
|     S001|  2026-07-28|       84|     4906.52|       40|2026-08-12 10:16:...|
|     S002|  2026-07-28|      168|     8485.40|       60|2026-08-12 10:16:...|
|     S003|  2026-07-28|      140|     7021.70|       54|2026-08-12 10:16:...|
|     S004|  2026-07-28|      116|     6767.90|       48|2026-08-12 10:16:...|
|     S005|  2026-07-28|      166|     9930.02|       64|2026-08-12 10:16:...|
+---------+------------+---------+------------+---------+--------------------+
only showing top 5 rows

+-----------+
|check_total|
+-----------+
|    8485.40|
+-----------+



In [16]:
#Writing a daily summary 

df_gold_daily_summary = (
    df_fact
    .withColumn("summary_date", F.to_date(F.col("trxn_dt")))
    .groupBy("summary_date")
    .agg(
        F.sum("amount").alias("total_revenue"),
        F.sum("qty").alias("total_quantity"),
        F.count("trxn_id").alias("total_transactions"),
        F.countDistinct("cust_num").alias("distinct_customers"),
        F.countDistinct("prod_num").alias("distinct_products"),
        F.countDistinct("store_num").alias("distinct_stores")
    )
    .orderBy("summary_date")
)
df_gold_daily_summary = df_gold_daily_summary.withColumn("gold_loaded_at", F.current_timestamp())
df_gold_daily_summary.show()

StatementMeta(, 2ec5e924-7aa4-4595-b3ed-29dece6154fd, 81, Finished, Available, Finished, False)

+------------+-------------+--------------+------------------+------------------+-----------------+---------------+--------------------+
|summary_date|total_revenue|total_quantity|total_transactions|distinct_customers|distinct_products|distinct_stores|      gold_loaded_at|
+------------+-------------+--------------+------------------+------------------+-----------------+---------------+--------------------+
|  2026-07-28|     37111.54|           674|               266|                98|               44|              5|2026-08-12 10:18:...|
|  2026-08-04|     40061.44|           793|               312|               142|               50|              5|2026-08-12 10:18:...|
|  2026-08-05|      5059.34|            97|                41|                39|               26|              5|2026-08-12 10:18:...|
|  2026-08-06|      5626.20|           104|                44|                42|               32|              5|2026-08-12 10:18:...|
|  2026-08-10|     13389.75|           24

In [17]:
df_gold_daily_summary.write.format("delta").mode("overwrite").saveAsTable("gold_daily_summary")

StatementMeta(, 2ec5e924-7aa4-4595-b3ed-29dece6154fd, 83, Finished, Available, Finished, False)